# Object-Oriented Programming in Python: Practice Set
**13 problems** from beginner to capstone, each with starter code, automated tests, a hint, and a collapsible reference solution.

## How to use this notebook
1. Read the problem statement.
2. Write your solution in the **starter cell** (replace the `pass` / `TODO`).
3. Run the **test cell** right below it. A line like `Problem 1 passed` means you are done; an `AssertionError` shows what to fix.
4. Stuck? Open the **Hint** first. When you are done (or give up), click **Show solution** under the test cell to compare. Solutions are collapsed so they don't spoil the problem.

> The test cells will fail until you implement the starter code. That is expected.

| Level | Problems | Focus |
|---|---|---|
| Easy | 1-3 | Classes, class vs instance attributes, properties |
| Medium | 4-8 | Exceptions, ABC, dunder methods, containers, dataclasses |
| Hard | 9-12 | MRO, context managers, iterators, design patterns |
| Capstone | 13 | Combine everything in one system |

**Suggested approach:** time yourself, write docstrings, and try to make each solution pass on the first run.

---
# Easy Level

## Problem 1: Rectangle
**Concepts:** classes, instance methods

Create a `Rectangle` class that takes `width` and `height`.

Implement:
- `area()` returns width x height
- `perimeter()` returns 2 x (width + height)
- `is_square()` returns `True` if width equals height

<details><summary><b>Hint</b></summary>

Store both values as instance attributes in `__init__`; every method just reads `self.width` and `self.height`.

</details>

In [ ]:
class Rectangle:
    """A rectangle with a width and a height."""
    # TODO: implement __init__, area, perimeter, is_square
    pass

In [ ]:
r = Rectangle(3, 4)
assert r.area() == 12
assert r.perimeter() == 14
assert r.is_square() is False
assert Rectangle(5, 5).is_square() is True
print("Problem 1 passed")

<details><summary><b>Show solution</b></summary>

```python
class Rectangle:
    """A rectangle with a width and a height."""

    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height

    def perimeter(self):
        return 2 * (self.width + self.height)

    def is_square(self):
        return self.width == self.height
```

</details>

## Problem 2: Ticket Counter
**Concepts:** class attributes vs instance attributes

Create a `Ticket` class that numbers tickets automatically.

- A **class attribute** `issued` tracks how many tickets have been created (starts at `0`).
- Each ticket gets an **instance attribute** `number`, starting at `1` for the first ticket, `2` for the second, and so on.

<details><summary><b>Hint</b></summary>

Increment through the class (`Ticket.issued += 1`), not through `self`, otherwise you create an instance attribute that shadows the shared one.

</details>

In [ ]:
class Ticket:
    # TODO: class attribute `issued`, instance attribute `number`
    pass

In [ ]:
Ticket.issued = 0          # reset so the test is repeatable
a, b, c = Ticket(), Ticket(), Ticket()
assert (a.number, b.number, c.number) == (1, 2, 3)
assert Ticket.issued == 3
assert "number" in vars(a) and "issued" not in vars(a)
print("Problem 2 passed")

<details><summary><b>Show solution</b></summary>

```python
class Ticket:
    issued = 0

    def __init__(self):
        Ticket.issued += 1
        self.number = Ticket.issued
```

</details>

## Problem 3: Thermometer
**Concepts:** properties, validation, read-only attributes

Create a `Thermometer` class.

- `celsius` is a property. Setting a value below **-273.15** must raise `ValueError` (including in `__init__`).
- `fahrenheit` is a **read-only** computed property: `celsius x 9/5 + 32`.
- `__init__(self, celsius=0.0)`

<details><summary><b>Hint</b></summary>

Store the real value in `_celsius`. Assign with `self.celsius = ...` in `__init__` so validation also runs there.

</details>

In [ ]:
class Thermometer:
    # TODO: celsius property with validation, read-only fahrenheit property
    pass

In [ ]:
t = Thermometer(100)
assert t.fahrenheit == 212
t.celsius = 0
assert t.fahrenheit == 32

for bad in (-300, -273.16):
    try:
        t.celsius = bad
        raise SystemError("ValueError expected")
    except ValueError:
        pass

try:
    Thermometer(-500)
    raise SystemError("ValueError expected in __init__")
except ValueError:
    pass

try:
    t.fahrenheit = 50
    raise SystemError("AttributeError expected")
except AttributeError:
    pass
print("Problem 3 passed")

<details><summary><b>Show solution</b></summary>

```python
class Thermometer:
    ABSOLUTE_ZERO = -273.15

    def __init__(self, celsius=0.0):
        self.celsius = celsius            # goes through the setter

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < self.ABSOLUTE_ZERO:
            raise ValueError("Below absolute zero")
        self._celsius = value

    @property
    def fahrenheit(self):
        return self._celsius * 9 / 5 + 32
```

</details>

---
# Medium Level

## Problem 4: Bank Account with Custom Exception
**Concepts:** custom exceptions, methods interacting with other objects

Build an `Account` class and a custom exception.

- `InsufficientFundsError(Exception)`
- `Account(owner, balance=0)` with a public `balance` attribute
- `deposit(amount)`: raises `ValueError` if `amount <= 0`
- `withdraw(amount)`: raises `InsufficientFundsError` if `amount > balance`
- `transfer(other, amount)`: moves money to another `Account`. If it fails, **neither** balance may change.

<details><summary><b>Hint</b></summary>

Do the withdrawal first. If it raises, the deposit never runs, so nothing is half-done.

</details>

In [ ]:
class InsufficientFundsError(Exception):
    pass


class Account:
    # TODO
    pass

In [ ]:
a, b = Account("Asha", 100), Account("Bilal")
a.transfer(b, 40)
assert (a.balance, b.balance) == (60, 40)

try:
    a.withdraw(1000)
    raise SystemError("InsufficientFundsError expected")
except InsufficientFundsError:
    pass

try:
    a.transfer(b, 1000)
    raise SystemError("InsufficientFundsError expected")
except InsufficientFundsError:
    pass
assert (a.balance, b.balance) == (60, 40)      # unchanged after failed transfer

for bad in (0, -5):
    try:
        a.deposit(bad)
        raise SystemError("ValueError expected")
    except ValueError:
        pass
print("Problem 4 passed")

<details><summary><b>Show solution</b></summary>

```python
class InsufficientFundsError(Exception):
    pass


class Account:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Deposit must be positive")
        self.balance += amount

    def withdraw(self, amount):
        if amount > self.balance:
            raise InsufficientFundsError(f"Balance {self.balance}, requested {amount}")
        self.balance -= amount

    def transfer(self, other, amount):
        self.withdraw(amount)       # raises before anything changes
        other.deposit(amount)
```

</details>

## Problem 5: Shapes with an Abstract Base Class
**Concepts:** ABC, abstract methods, polymorphism

Create an abstract class `Shape` with an abstract method `area()`.

Then implement `Circle(radius)` and `Rectangle(width, height)`, and a function `total_area(shapes)` that returns the sum of all areas.

`Shape()` itself must not be instantiable.

<details><summary><b>Hint</b></summary>

`Shape` must inherit from `ABC` and decorate `area` with `@abstractmethod`. `total_area` needs no `isinstance` checks.

</details>

In [ ]:
from abc import ABC, abstractmethod
import math

# TODO: Shape, Circle, Rectangle, total_area

In [ ]:
try:
    Shape()
    raise SystemError("TypeError expected")
except TypeError:
    pass

shapes = [Circle(1), Rectangle(2, 3)]
assert abs(total_area(shapes) - (math.pi + 6)) < 1e-9
assert all(isinstance(s, Shape) for s in shapes)
assert total_area([]) == 0
print("Problem 5 passed")

<details><summary><b>Show solution</b></summary>

```python
from abc import ABC, abstractmethod
import math


class Shape(ABC):
    @abstractmethod
    def area(self) -> float:
        ...


class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return math.pi * self.radius ** 2


class Rectangle(Shape):
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height


def total_area(shapes):
    return sum(s.area() for s in shapes)
```

</details>

## Problem 6: Vector2D Operators
**Concepts:** dunder methods, operator overloading, hashing

Implement `Vector2D(x, y)` supporting:

| Expression | Result |
|---|---|
| `v + w`, `v - w` | component-wise `Vector2D` |
| `v * 2` **and** `2 * v` | scaled `Vector2D` |
| `abs(v)` | length of the vector |
| `v == w` | equal when both components are equal |
| `repr(v)` | `Vector2D(3, 4)` |
| `set` / `dict` use | equal vectors must hash equally |

<details><summary><b>Hint</b></summary>

`2 * v` calls `int.__mul__` first, which fails, then falls back to `v.__rmul__`. Defining `__eq__` removes the default `__hash__`, so add one.

</details>

In [ ]:
class Vector2D:
    # TODO
    pass

In [ ]:
v, w = Vector2D(3, 4), Vector2D(1, 2)
assert v + w == Vector2D(4, 6)
assert v - w == Vector2D(2, 2)
assert v * 2 == Vector2D(6, 8)
assert 2 * v == Vector2D(6, 8)
assert abs(v) == 5
assert repr(v) == "Vector2D(3, 4)"
assert len({Vector2D(1, 2), Vector2D(1, 2)}) == 1
assert (v == "not a vector") is False
print("Problem 6 passed")

<details><summary><b>Show solution</b></summary>

```python
import math


class Vector2D:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __repr__(self):
        return f"Vector2D({self.x}, {self.y})"

    def __add__(self, other):
        return Vector2D(self.x + other.x, self.y + other.y)

    def __sub__(self, other):
        return Vector2D(self.x - other.x, self.y - other.y)

    def __mul__(self, k):
        return Vector2D(self.x * k, self.y * k)

    __rmul__ = __mul__

    def __abs__(self):
        return math.hypot(self.x, self.y)

    def __eq__(self, other):
        return isinstance(other, Vector2D) and (self.x, self.y) == (other.x, other.y)

    def __hash__(self):
        return hash((self.x, self.y))
```

</details>

## Problem 7: Stack
**Concepts:** container protocol, exceptions

Implement a `Stack` class.

- `push(item)`, `pop()`, `peek()`
- `pop()` and `peek()` raise `IndexError` when the stack is empty
- `len(stack)` returns the number of items
- Iterating (`for x in stack`) yields items from **top to bottom**

<details><summary><b>Hint</b></summary>

Keep a private list. `reversed(self._items)` gives top-to-bottom order for `__iter__`.

</details>

In [ ]:
class Stack:
    # TODO
    pass

In [ ]:
s = Stack()
assert len(s) == 0
for i in (1, 2, 3):
    s.push(i)
assert s.peek() == 3
assert list(s) == [3, 2, 1]
assert s.pop() == 3 and len(s) == 2
s.pop(); s.pop()

for op in (s.pop, s.peek):
    try:
        op()
        raise SystemError("IndexError expected")
    except IndexError:
        pass
print("Problem 7 passed")

<details><summary><b>Show solution</b></summary>

```python
class Stack:
    def __init__(self):
        self._items = []

    def push(self, item):
        self._items.append(item)

    def pop(self):
        if not self._items:
            raise IndexError("pop from empty stack")
        return self._items.pop()

    def peek(self):
        if not self._items:
            raise IndexError("peek from empty stack")
        return self._items[-1]

    def __len__(self):
        return len(self._items)

    def __iter__(self):
        return iter(reversed(self._items))
```

</details>

## Problem 8: Inventory (Dataclass + Composition)
**Concepts:** dataclasses, composition

1. Create a dataclass `Item(name, price, quantity=0)`.
2. Create `Inventory` that **has** a dict `items` mapping `name -> Item`.
   - `add_item(item)`: if the name already exists, **add** to its quantity; otherwise store the item.
   - `total_value()`: sum of `price x quantity` over all items.
   - `low_stock(threshold=5)`: sorted list of names whose quantity is **strictly below** `threshold`.

<details><summary><b>Hint</b></summary>

Use `self.items = {}` inside `__init__` (not as a class attribute) so each inventory has its own dict.

</details>

In [ ]:
from dataclasses import dataclass

# TODO: Item (dataclass) and Inventory

In [ ]:
inv = Inventory()
inv.add_item(Item("pen", 10, 3))
inv.add_item(Item("book", 200, 10))
inv.add_item(Item("pen", 10, 2))
assert inv.items["pen"].quantity == 5
assert inv.total_value() == 5 * 10 + 10 * 200
assert inv.low_stock(6) == ["pen"]
assert inv.low_stock(5) == []
assert Item("x", 1) == Item("x", 1, 0)
print("Problem 8 passed")

<details><summary><b>Show solution</b></summary>

```python
from dataclasses import dataclass


@dataclass
class Item:
    name: str
    price: float
    quantity: int = 0


class Inventory:
    def __init__(self):
        self.items = {}

    def add_item(self, item):
        if item.name in self.items:
            self.items[item.name].quantity += item.quantity
        else:
            self.items[item.name] = item

    def total_value(self):
        return sum(i.price * i.quantity for i in self.items.values())

    def low_stock(self, threshold=5):
        return sorted(n for n, i in self.items.items() if i.quantity < threshold)
```

</details>

---
# Hard Level

## Problem 9: Predict the MRO
**Concepts:** multiple inheritance, C3 linearisation

Read the classes below **without running them**, then set `expected_mro` to the list of class names in the order Python searches them for `D().greet`.

Include `"object"` at the end.

<details><summary><b>Hint</b></summary>

Parents are searched left to right (`C` before `B`), but a shared parent (`A`) always comes after **all** its children.

</details>

In [ ]:
class A:
    def greet(self): return "A"

class B(A):
    def greet(self): return "B>" + super().greet()

class C(A):
    def greet(self): return "C>" + super().greet()

class D(C, B):
    def greet(self): return "D>" + super().greet()


expected_mro = []      # TODO: e.g. ["D", "...", "object"]

In [ ]:
assert expected_mro == [cls.__name__ for cls in D.__mro__], "Check your order against D.__mro__"
assert D().greet() == "D>C>B>A"
print("Problem 9 passed")

<details><summary><b>Show solution</b></summary>

```python
class A:
    def greet(self): return "A"

class B(A):
    def greet(self): return "B>" + super().greet()

class C(A):
    def greet(self): return "C>" + super().greet()

class D(C, B):
    def greet(self): return "D>" + super().greet()


expected_mro = ["D", "C", "B", "A", "object"]
```

</details>

## Problem 10: Rollback Transaction
**Concepts:** context managers

Write a context manager class `Transaction(data)` that protects a dictionary.

- On entering, take a snapshot of `data` and return `data`.
- If the `with` block finishes normally, keep the changes.
- If an exception occurs, **restore** `data` to the snapshot and let the exception propagate.

<details><summary><b>Hint</b></summary>

Use `copy.deepcopy` so nested lists are protected too. Returning `False` from `__exit__` lets the exception continue.

</details>

In [ ]:
import copy

class Transaction:
    # TODO: __init__, __enter__, __exit__
    pass

In [ ]:
data = {"balance": 100, "log": []}

with Transaction(data) as d:
    d["balance"] = 50
    d["log"].append("ok")
assert data == {"balance": 50, "log": ["ok"]}

try:
    with Transaction(data) as d:
        d["balance"] = 0
        d["log"].append("bad")
        raise RuntimeError("boom")
except RuntimeError:
    pass
assert data == {"balance": 50, "log": ["ok"]}      # rolled back
print("Problem 10 passed")

<details><summary><b>Show solution</b></summary>

```python
import copy


class Transaction:
    def __init__(self, data):
        self.data = data

    def __enter__(self):
        self._snapshot = copy.deepcopy(self.data)
        return self.data

    def __exit__(self, exc_type, exc_value, traceback):
        if exc_type is not None:
            self.data.clear()
            self.data.update(self._snapshot)
        return False         # never swallow the exception
```

</details>

## Problem 11: Fibonacci Iterator
**Concepts:** iterator protocol

Implement `Fibonacci(n)` as a class-based **iterator** that yields the first `n` Fibonacci numbers (`0, 1, 1, 2, 3, 5, ...`).

Do not use generators (`yield`); implement `__iter__` and `__next__` yourself.

<details><summary><b>Hint</b></summary>

`__iter__` returns `self`. `__next__` returns the current value, advances the state, and raises `StopIteration` when done.

</details>

In [ ]:
class Fibonacci:
    # TODO: __init__, __iter__, __next__
    pass

In [ ]:
assert list(Fibonacci(8)) == [0, 1, 1, 2, 3, 5, 8, 13]
assert list(Fibonacci(0)) == []
fib = Fibonacci(3)
assert next(fib) == 0 and next(fib) == 1 and next(fib) == 1
try:
    next(fib)
    raise SystemError("StopIteration expected")
except StopIteration:
    pass
print("Problem 11 passed")

<details><summary><b>Show solution</b></summary>

```python
class Fibonacci:
    def __init__(self, n):
        self.n = n
        self._count = 0
        self._a, self._b = 0, 1

    def __iter__(self):
        return self

    def __next__(self):
        if self._count >= self.n:
            raise StopIteration
        value = self._a
        self._a, self._b = self._b, self._a + self._b
        self._count += 1
        return value
```

</details>

## Problem 12: Discount Strategies
**Concepts:** Strategy pattern, ABC, dependency inversion

Build a shopping cart whose discount logic can be swapped without changing the cart.

- Abstract `Discount` with `apply(total) -> float`
- `NoDiscount`, `PercentageDiscount(percent)`, `FlatDiscount(amount)` (never returns below `0`)
- `Cart(discount=None)` with `add(name, price, qty=1)` and `total()`; the default discount is `NoDiscount`
- The `discount` attribute can be replaced at any time

<details><summary><b>Hint</b></summary>

The cart never checks which discount it has; it only calls `self.discount.apply(subtotal)`. Avoid `discount=NoDiscount()` as a default argument.

</details>

In [ ]:
from abc import ABC, abstractmethod

# TODO: Discount, NoDiscount, PercentageDiscount, FlatDiscount, Cart

In [ ]:
cart = Cart()
cart.add("pen", 10.0, 2)
cart.add("book", 5.0)
assert cart.total() == 25.0

cart.discount = PercentageDiscount(10)
assert abs(cart.total() - 22.5) < 1e-9

cart.discount = FlatDiscount(5)
assert cart.total() == 20.0

cart.discount = FlatDiscount(100)
assert cart.total() == 0

try:
    Discount()
    raise SystemError("TypeError expected")
except TypeError:
    pass
print("Problem 12 passed")

<details><summary><b>Show solution</b></summary>

```python
from abc import ABC, abstractmethod


class Discount(ABC):
    @abstractmethod
    def apply(self, total: float) -> float:
        ...


class NoDiscount(Discount):
    def apply(self, total):
        return total


class PercentageDiscount(Discount):
    def __init__(self, percent):
        self.percent = percent

    def apply(self, total):
        return total * (1 - self.percent / 100)


class FlatDiscount(Discount):
    def __init__(self, amount):
        self.amount = amount

    def apply(self, total):
        return max(total - self.amount, 0)


class Cart:
    def __init__(self, discount=None):
        self.discount = discount or NoDiscount()
        self._lines = []

    def add(self, name, price, qty=1):
        self._lines.append((name, price, qty))

    def total(self):
        subtotal = sum(price * qty for _, price, qty in self._lines)
        return self.discount.apply(subtotal)
```

</details>

---
# Capstone Level

## Problem 13: Library Management System
**Concepts:** everything combined

Design a small library system.

**Exceptions:** `LibraryError(Exception)`, with subclasses `BookUnavailableError` and `LimitExceededError`.

**Classes**
- `Book` is a dataclass: `isbn`, `title`, `author`, `available=True`
- `Member(name)`: has a `borrowed` list and class attribute `MAX_BOOKS = 3`
- `Library`:
  - `add_book(book)`
  - `checkout(isbn, member)`: raises `KeyError` for an unknown ISBN, `BookUnavailableError` if already taken, `LimitExceededError` if the member already holds `MAX_BOOKS`. On success mark the book unavailable and add it to `member.borrowed`.
  - `return_book(isbn, member)`: marks the book available again and removes it from the member (raises `ValueError` if the member doesn't hold it)
  - `search(author)`: sorted list of titles by that author

<details><summary><b>Hint</b></summary>

Validate first, mutate last: raise all errors before changing any state. `list.remove` already raises `ValueError` for you.

</details>

In [ ]:
from dataclasses import dataclass

# TODO: exceptions, Book, Member, Library

In [ ]:
lib = Library()
for i, (title, author) in enumerate([("A1", "Ann"), ("A2", "Ann"), ("B1", "Bob"), ("C1", "Cy"), ("D1", "Di")]):
    lib.add_book(Book(str(i), title, author))

m, n = Member("Mira"), Member("Noor")
assert lib.search("Ann") == ["A1", "A2"]

lib.checkout("0", m)
assert m.borrowed[0].title == "A1"

try:
    lib.checkout("0", n)
    raise SystemError("BookUnavailableError expected")
except BookUnavailableError:
    pass

lib.checkout("1", m); lib.checkout("2", m)
try:
    lib.checkout("3", m)
    raise SystemError("LimitExceededError expected")
except LimitExceededError as err:
    assert isinstance(err, LibraryError)

try:
    lib.checkout("999", m)
    raise SystemError("KeyError expected")
except KeyError:
    pass

lib.return_book("0", m)
assert len(m.borrowed) == 2
lib.checkout("0", n)                       # available again

try:
    lib.return_book("4", m)                # m never borrowed this one
    raise SystemError("ValueError expected")
except ValueError:
    pass
print("Problem 13 (Capstone) passed")

<details><summary><b>Show solution</b></summary>

```python
from dataclasses import dataclass


class LibraryError(Exception):
    pass


class BookUnavailableError(LibraryError):
    pass


class LimitExceededError(LibraryError):
    pass


@dataclass
class Book:
    isbn: str
    title: str
    author: str
    available: bool = True


class Member:
    MAX_BOOKS = 3

    def __init__(self, name):
        self.name = name
        self.borrowed = []


class Library:
    def __init__(self):
        self._books = {}

    def add_book(self, book):
        self._books[book.isbn] = book

    def checkout(self, isbn, member):
        book = self._books[isbn]                       # KeyError if unknown
        if not book.available:
            raise BookUnavailableError(book.title)
        if len(member.borrowed) >= member.MAX_BOOKS:
            raise LimitExceededError(member.name)
        book.available = False
        member.borrowed.append(book)

    def return_book(self, isbn, member):
        book = self._books[isbn]
        member.borrowed.remove(book)                   # ValueError if not held
        book.available = True

    def search(self, author):
        return sorted(b.title for b in self._books.values() if b.author == author)
```

</details>

---
## Scorecard
Tick off each problem once its test passes.

- [ ] 1 Rectangle
- [ ] 2 Ticket Counter
- [ ] 3 Thermometer
- [ ] 4 Bank Account
- [ ] 5 Shapes (ABC)
- [ ] 6 Vector2D
- [ ] 7 Stack
- [ ] 8 Inventory
- [ ] 9 Predict the MRO
- [ ] 10 Transaction
- [ ] 11 Fibonacci Iterator
- [ ] 12 Discount Strategies
- [ ] 13 Library Capstone

### Stretch goals
- Add type hints and docstrings to every solution.
- Rewrite Problem 4 using `@property` for `balance`.
- Rewrite Problem 11 as a generator function with `yield` and compare.
- Add `__str__`, `__lt__` and `@total_ordering` to `Book` in the capstone.